# Avaliação automática de detecção e alertas em texto

Este notebook substitui o **fluxo de preenchimento manual** dos notebooks 05, 05b e 05c.
Não lê nenhuma planilha manual. Usa as imagens e os arquivos de anotação `.txt` do
TEST existente. Os CSVs gerados são **saídas**, e não entradas a preencher.

**Execução:** Windows + Jupyter, no ambiente já utilizado no projeto. Não há treinamento
neste notebook. O modelo atual e as 403 imagens do TEST são preservados. A saída final
da aplicação é uma mensagem de alerta **em texto**, quando a política exigir.

## O que esta avaliação mede

1. **Detector:** Precision, Recall, mAP@50 e mAP@50–95 contra as anotações do dataset.
2. **Alertas:** efeito dos erros de detecção sobre uma política fixa. A referência
   automática é a política aplicada às caixas e classes anotadas. Ela **não é um
   gabarito independente de risco físico**. Concordância com essa referência não
   comprova segurança ou risco real de locomoção.

Todas as imagens entram na avaliação, inclusive as que contêm detecções incorretas,
objetos não detectados ou nenhuma previsão. Não existe seleção manual de 50 cenários.
Os 50 cenários antigos permanecem como resultado histórico; não são necessários aqui.

A regra usa as constantes e funções da **V2 original anexada**, centralizadas em
`src/politica_alertas.py`, incluindo os terços horizontais e o piso de pontuação.
As classes contextuais e ignoradas continuam identificadas internamente, sem alertas.

Não ajuste a política, a confiança ou o IoU a partir dos resultados do TEST. Qualquer
desenvolvimento posterior deve usar dados de desenvolvimento e guardar a configuração
de cada experimento. Não compare estes resultados de política diretamente com a
acurácia de risco humano obtida nos 50 cenários antigos.

## Como instalar no projeto

1. Instale primeiro os módulos `src` deste pacote e salve este arquivo em `tcc_visao_computacional/notebooks/`.
2. Mantenha o modelo em `models/yolov8n_baseline/best.pt` e a base em
   `data/processed/homeobjects_yolo/`.
3. Abra este notebook no Jupyter do mesmo ambiente utilizado nos notebooks anteriores.
4. Execute as células na ordem. Configure caminhos somente se a localização for diferente.
5. Os resultados vão para uma nova pasta em `reports/avaliacao_automatica/` a cada execução.

Documentação oficial:
- [Avaliação YOLO](https://docs.ultralytics.com/modes/val/)
- [Inferência e caixas previstas](https://docs.ultralytics.com/modes/predict/)
- [HomeObjects-3K](https://docs.ultralytics.com/datasets/detect/homeobjects-3k/)


## 1. Dependências

Use inicialmente o ambiente atual, sem atualizar seus pacotes. A célula seguinte está
comentada para evitar mudanças desnecessárias. Se a importação da próxima célula acusar
um pacote ausente, retire o `#` da instalação, execute e reinicie o kernel.

A instalação exige acesso à internet. Todas as versões utilizadas serão registradas.


In [ ]:
# Descomente apenas se faltar algum pacote no seu ambiente:
# %pip install ultralytics pandas numpy matplotlib pillow pyyaml


In [ ]:
import hashlib
import json
import math
import platform
import time
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

def versao_pacote(nome):
    try:
        return version(nome)
    except PackageNotFoundError:
        return "não identificada"

print("Python:", platform.python_version())
print("Ultralytics:", versao_pacote("ultralytics"))


## 2. Configuração única

Isto é configuração do projeto, não anotação de imagens. `RAIZ_PROJETO = None` procura
a pasta automaticamente a partir do diretório do Jupyter. Caso a busca falhe, informe
o caminho da pasta `tcc_visao_computacional`, por exemplo
`Path(r"C:\Users\Diego\Documents\tcc_visao_computacional")`.

O alerta textual será emitido para prioridade **média ou alta**. Prioridade baixa e
classes fora da política não acionam mensagens. Essa decisão operacional é explícita
e pode ser revisada com dados de desenvolvimento, antes de uma nova avaliação final.

`CONF_ALERTA = 0.25` é a confiança usada na aplicação. O cálculo de mAP usa confiança
baixa (`0.001`) para considerar a curva de desempenho; são avaliações diferentes.


In [ ]:
RAIZ_PROJETO = None  # ou Path(r"C:\Users\Diego\Documents\tcc_visao_computacional")
DISPOSITIVO = "cpu"  # no Colab com GPU disponível, pode ser 0
BATCH_AVALIACAO = 4  # conservador para inferência no Windows
IMGSZ = 640
CONF_ALERTA = 0.25
IOU_NMS = 0.70
IOU_CORRESPONDENCIA = 0.50
MAX_DETECCOES = 300
NUMERO_ESPERADO_IMAGENS = 403
NIVEL_MINIMO_ALERTA = "medio"

CLASSES_ESPERADAS = [
    "bed", "sofa", "chair", "table", "lamp", "tv", "laptop",
    "wardrobe", "window", "door", "potted plant", "photo frame",
]

if RAIZ_PROJETO is None:
    candidatos = [Path.cwd(), *Path.cwd().parents]
    RAIZ_PROJETO = next((p for p in candidatos
                        if (p / "data/processed/homeobjects_yolo/data.yaml").is_file()
                        and (p / "models/yolov8n_baseline/best.pt").is_file()), None)
    if RAIZ_PROJETO is None:
        raise FileNotFoundError(
            "Não encontrei a raiz. Informe RAIZ_PROJETO nesta célula e execute novamente."
        )
RAIZ_PROJETO = Path(RAIZ_PROJETO).expanduser().resolve()
BASE_DATASET = RAIZ_PROJETO / "data/processed/homeobjects_yolo"
CAMINHO_MODELO = RAIZ_PROJETO / "models/yolov8n_baseline/best.pt"
CAMINHO_YAML_ORIGINAL = BASE_DATASET / "data.yaml"
PASTA_IMAGENS = BASE_DATASET / "images/test"
PASTA_LABELS = BASE_DATASET / "labels/test"

for caminho in [CAMINHO_MODELO, CAMINHO_YAML_ORIGINAL, PASTA_IMAGENS, PASTA_LABELS]:
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo ou pasta não encontrado: {caminho}")
if NIVEL_MINIMO_ALERTA not in {"medio", "alto"}:
    raise ValueError("NIVEL_MINIMO_ALERTA deve ser medio ou alto.")
print("Projeto:", RAIZ_PROJETO)
print("Modelo:", CAMINHO_MODELO)
print("TEST:", PASTA_IMAGENS)

import sys
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from src.politica_alertas import (
    PESO_OBJETO, CLASSES_CONTEXTUAIS, CLASSES_IGNORADAS,
    LIMITE_ESQUERDA, LIMITE_DIREITA, LIMITE_AREA_MEDIA, LIMITE_AREA_GRANDE,
    LIMITE_VERTICAL_SUPERIOR, LIMITE_VERTICAL_INFERIOR,
    PESO_POSICAO, PESO_TAMANHO, AJUSTE_VERTICAL,
    classificar_posicao, classificar_tamanho, classificar_posicao_vertical,
    classificar_risco_v1, classificar_risco_v2, gerar_mensagens_tabela,
)

from src import politica_alertas as politica_original
from src.preparacao_dados import resumo_test, salvar_json as salvar_json_preparacao

POLITICA = {
    'versao': 'v2_original_congelada',
    'pesos_classe': PESO_OBJETO, 'limites_horizontais': [LIMITE_ESQUERDA, LIMITE_DIREITA],
    'pesos_horizontais': PESO_POSICAO, 'limites_area': [LIMITE_AREA_MEDIA, LIMITE_AREA_GRANDE],
    'pesos_tamanho': PESO_TAMANHO,
    'limites_base_vertical': [LIMITE_VERTICAL_SUPERIOR, LIMITE_VERTICAL_INFERIOR],
    'ajustes_verticais': AJUSTE_VERTICAL, 'limite_medio': 5, 'limite_alto': 7, 'piso_pontuacao': 1,
}


## 3. Ler e conferir as anotações existentes

O código lê o `data.yaml` para obter os IDs das classes, abre cada imagem e converte
as caixas YOLO para coordenadas em pixels. Interrompe a execução em caso de imagens
corrompidas, labels ausentes, valores inválidos ou número de imagens diferente de 403.
Não exclui casos problemáticos silenciosamente e não solicita preencher planilhas.

Um label vazio existente representa imagem sem objeto anotado. Um label ausente
representa problema estrutural e não é tratado como imagem negativa.


In [ ]:
def sha256_arquivo(caminho):
    digest = hashlib.sha256()
    with Path(caminho).open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            digest.update(bloco)
    return digest.hexdigest()

def normalizar_nomes(nomes):
    if isinstance(nomes, list):
        return {i: str(nome) for i, nome in enumerate(nomes)}
    if isinstance(nomes, dict):
        return {int(i): str(nome) for i, nome in nomes.items()}
    raise ValueError("names deve ser uma lista ou um dicionário de classes.")

def ler_objetos_yolo(caminho, largura, altura, nomes):
    objetos = []
    for linha_n, linha in enumerate(Path(caminho).read_text(encoding="utf-8-sig").splitlines(), 1):
        if not linha.strip():
            continue
        partes = linha.split()
        if len(partes) != 5:
            raise ValueError(f"{caminho}, linha {linha_n}: esperado classe xc yc w h.")
        valores = [float(p) for p in partes]
        if not all(math.isfinite(v) for v in valores):
            raise ValueError(f"{caminho}, linha {linha_n}: valor não finito.")
        id_float, xc, yc, w, h = valores
        classe_id = int(id_float)
        if id_float != classe_id or classe_id not in nomes or w <= 0 or h <= 0:
            raise ValueError(f"{caminho}, linha {linha_n}: classe ou tamanho inválido.")
        bordas = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]
        if min(bordas) < -0.0001 or max(bordas) > 1.0001:
            raise ValueError(f"{caminho}, linha {linha_n}: caixa fora da imagem.")
        x1, y1, x2, y2 = np.clip(bordas, 0.0, 1.0)
        objetos.append({
            "classe_id": classe_id, "classe": nomes[classe_id],
            "bbox": [float(x1 * largura), float(y1 * altura),
                     float(x2 * largura), float(y2 * altura)],
            "confianca": None,
        })
    return objetos

yaml_original = yaml.safe_load(CAMINHO_YAML_ORIGINAL.read_text(encoding="utf-8-sig"))
NOMES = normalizar_nomes(yaml_original["names"])
if set(NOMES) != set(range(len(NOMES))) or set(NOMES.values()) != set(CLASSES_ESPERADAS):
    raise ValueError("As classes do data.yaml não correspondem às 12 classes do contexto.")

extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
imagens = sorted(p for p in PASTA_IMAGENS.rglob("*") if p.suffix.lower() in extensoes)
if len(imagens) != NUMERO_ESPERADO_IMAGENS:
    raise ValueError(f"TEST com {len(imagens)} imagens; esperado {NUMERO_ESPERADO_IMAGENS}.")

REGISTROS = []
manifesto = []
for caminho in imagens:
    relativo = caminho.relative_to(PASTA_IMAGENS)
    label = PASTA_LABELS / relativo.with_suffix(".txt")
    if not label.is_file():
        raise FileNotFoundError(f"Anotação ausente: {label}")
    with Image.open(caminho) as imagem:
        largura, altura = imagem.size
        imagem.verify()
    objetos = ler_objetos_yolo(label, largura, altura, NOMES)
    REGISTROS.append({"imagem": caminho, "nome": relativo.as_posix(),
                      "largura": largura, "altura": altura, "gt": objetos})
    manifesto.append({"imagem": relativo.as_posix(),
                      "imagem_sha256": sha256_arquivo(caminho),
                      "label_sha256": sha256_arquivo(label), "objetos": len(objetos)})

print("Imagens conferidas:", len(REGISTROS))
print("Objetos anotados:", sum(len(r["gt"]) for r in REGISTROS))
print("Sem preenchimento manual: a referência foi lida dos labels existentes.")

# A mesma proteção usada no notebook 02; inclui nomes, imagens, labels e mapa de classes.
manifesto_protecao = pd.DataFrame([
    {'split': 'test', 'relativo': r['imagem'], 'sha256': r['imagem_sha256'],
     'label_sha256': r['label_sha256']} for r in manifesto
])
snapshot_test = resumo_test(manifesto_protecao, NOMES, NUMERO_ESPERADO_IMAGENS)
trava_test = RAIZ_PROJETO / 'reports/protecao_test/test_congelado.json'
if trava_test.exists():
    if json.loads(trava_test.read_text(encoding='utf-8'))['sha256'] != snapshot_test['sha256']:
        raise RuntimeError('O TEST foi alterado desde a referência congelada. Avaliação bloqueada.')
else:
    salvar_json_preparacao(trava_test, snapshot_test)


## 4. Política fixa e mensagens em texto

A política aplica classe + posição horizontal + tamanho relativo + ajuste vertical.
Pontuações menores que 5 são baixas, de 5 a 6 médias e a partir de 7 altas. Os ajustes
verticais podem produzir pontuação menor que 3; ela continua na faixa baixa.

Somente as seis classes de obstáculos do contexto participam dos alertas. `lamp` e
`door`, assim como as demais classes fora da lista de pesos, continuam na avaliação
do detector, mas não geram mensagem por esta política.

As mensagens descrevem a **posição na imagem**. Não afirmam distância em metros,
contato com o chão ou existência de um trajeto seguro.


In [ ]:
ORDEM_PRIORIDADE = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}
NOMES_PT = {
    "bed": "cama", "sofa": "sofá", "chair": "cadeira", "table": "mesa",
    "wardrobe": "guarda-roupa", "potted plant": "planta em vaso",
}
VERBOS_PT = {"bed": "detectada", "sofa": "detectado", "chair": "detectada",
             "table": "detectada", "wardrobe": "detectado", "potted plant": "detectada"}
POSICOES_PT = {"esquerda": "esquerda", "centro": "central", "direita": "direita"}
PRIORIDADES_PT = {"baixo": "baixa", "medio": "média", "alto": "alta"}

def avaliar_prioridade(objeto, largura, altura):
    return politica_original.avaliar_bbox(objeto['classe'], objeto['bbox'], largura, altura)


def aciona_alerta(prioridade):
    return ORDEM_PRIORIDADE[prioridade] >= ORDEM_PRIORIDADE[NIVEL_MINIMO_ALERTA]

def adicionar_prioridades(objetos, largura, altura):
    return [{**objeto, **avaliar_prioridade(objeto, largura, altura)} for objeto in objetos]

def gerar_mensagem(objeto):
    return politica_original.gerar_mensagem(objeto, NIVEL_MINIMO_ALERTA)


def prioridade_imagem(objetos):
    return max((o["prioridade"] for o in objetos),
               key=lambda p: ORDEM_PRIORIDADE[p], default="sem_prioridade")

def objetos_resultado(resultado):
    if resultado.boxes is None or len(resultado.boxes) == 0:
        return []
    caixas = resultado.boxes.xyxy.cpu().numpy()
    classes = resultado.boxes.cls.cpu().numpy().astype(int)
    confiancas = resultado.boxes.conf.cpu().numpy()
    return [{"classe_id": int(c), "classe": NOMES[int(c)],
             "bbox": [float(v) for v in caixa], "confianca": float(conf)}
            for caixa, c, conf in zip(caixas, classes, confiancas)]

def prever_imagem(modelo, caminho):
    return modelo.predict(source=str(caminho), imgsz=IMGSZ, conf=CONF_ALERTA,
                          iou=IOU_NMS, max_det=MAX_DETECCOES,
                          device=DISPOSITIVO, verbose=False, save=False)[0]

print("Política:", POLITICA["versao"])
print("Emitir mensagem a partir de:", NIVEL_MINIMO_ALERTA)


## 5. Associação automática entre objetos anotados e previstos

Uma correspondência exige **mesma classe e IoU >= 0,50**. Cada anotação e cada previsão
participam de, no máximo, uma correspondência. O algoritmo usa as maiores sobreposições
primeiro. Previsões extras e anotações sem previsão permanecem no relatório.

Uma classe incorreta conta como previsão não associada e objeto anotado sem detecção.
Este diagnóstico em confiança fixa complementa, e não substitui, o mAP oficial.


In [ ]:
def iou_matriz(gt, pred):
    if not gt or not pred:
        return np.zeros((len(gt), len(pred)), dtype=float)
    g = np.asarray([o["bbox"] for o in gt], dtype=float)
    p = np.asarray([o["bbox"] for o in pred], dtype=float)
    inicio = np.maximum(g[:, None, :2], p[None, :, :2])
    fim = np.minimum(g[:, None, 2:], p[None, :, 2:])
    inter = np.prod(np.maximum(0.0, fim - inicio), axis=2)
    ag = np.prod(np.maximum(0.0, g[:, 2:] - g[:, :2]), axis=1)
    ap = np.prod(np.maximum(0.0, p[:, 2:] - p[:, :2]), axis=1)
    uniao = ag[:, None] + ap[None, :] - inter
    return np.divide(inter, uniao, out=np.zeros_like(inter), where=uniao > 0)

def corresponder_objetos(gt, pred):
    matriz = iou_matriz(gt, pred)
    candidatos = []
    for gi, g in enumerate(gt):
        for pi, p in enumerate(pred):
            if g["classe_id"] == p["classe_id"] and matriz[gi, pi] >= IOU_CORRESPONDENCIA:
                candidatos.append((float(matriz[gi, pi]), gi, pi))
    candidatos.sort(key=lambda item: (-item[0], item[1], item[2]))
    usados_g, usados_p, pares = set(), set(), []
    for iou, gi, pi in candidatos:
        if gi not in usados_g and pi not in usados_p:
            pares.append((gi, pi, iou))
            usados_g.add(gi)
            usados_p.add(pi)
    sem_gt = [i for i in range(len(gt)) if i not in usados_g]
    sem_pred = [i for i in range(len(pred)) if i not in usados_p]
    return pares, sem_gt, sem_pred

def linha_objeto(nome, evento, gi=None, pi=None, g=None, p=None, iou=None):
    pg = g["prioridade"] if g else "sem_prioridade"
    pp = p["prioridade"] if p else "sem_prioridade"
    return {
        "imagem": nome, "evento": evento, "id_gt": gi, "id_pred": pi,
        "classe_ref": g["classe"] if g else None,
        "classe_pred": p["classe"] if p else None,
        "iou": iou, "confianca": p["confianca"] if p else None,
        "bbox_ref": json.dumps(g["bbox"]) if g else None,
        "bbox_pred": json.dumps(p["bbox"]) if p else None,
        "prioridade_ref_politica": pg, "prioridade_pred": pp,
        "ha_alerta_ref_politica": bool(g and aciona_alerta(pg)),
        "ha_alerta_pred": bool(p and aciona_alerta(pp)),
        "concordancia_prioridade": pg == pp if g and p else None,
    }

print("Associação automática configurada.")


## 6. Carregar o modelo e registrar o experimento

Esta célula verifica se o modelo usa exatamente o mesmo mapa de IDs do dataset.
Cria uma pasta nova para não misturar resultados de execuções diferentes. Registra
versões, configuração, hashes dos arquivos de teste e hash do modelo para rastreabilidade.


In [ ]:
modelo = YOLO(str(CAMINHO_MODELO))
if normalizar_nomes(modelo.names) != NOMES:
    raise ValueError("IDs/classes do modelo e do dataset são diferentes. Não remapeie o TEST para corrigir isso.")

identificador = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%f")
PASTA_SAIDA = RAIZ_PROJETO / "reports/avaliacao_automatica" / identificador
PASTA_SAIDA.mkdir(parents=True, exist_ok=False)

def salvar_json(caminho, conteudo):
    Path(caminho).write_text(json.dumps(conteudo, ensure_ascii=False, indent=2,
                                      allow_nan=False), encoding="utf-8")

configuracao = {
    "criado_em": datetime.now().astimezone().isoformat(),
    "python": platform.python_version(),
    "plataforma": platform.platform(),
    "pacotes": {p: versao_pacote(p) for p in
                ["ultralytics", "torch", "numpy", "pandas", "matplotlib", "Pillow", "PyYAML"]},
    "modelo": str(CAMINHO_MODELO), "modelo_sha256": sha256_arquivo(CAMINHO_MODELO),
    "dataset": str(BASE_DATASET), "classes": NOMES,
    "imagens_test": len(REGISTROS),
    "objetos_test": sum(len(r["gt"]) for r in REGISTROS),
    "test_sha256": hashlib.sha256(json.dumps(manifesto, sort_keys=True).encode()).hexdigest(),
    "imgsz": IMGSZ, "device": DISPOSITIVO, "batch_avaliacao": BATCH_AVALIACAO,
    "conf_alerta": CONF_ALERTA, "conf_map": 0.001,
    "iou_nms": IOU_NMS, "iou_correspondencia": IOU_CORRESPONDENCIA,
    "max_deteccoes": MAX_DETECCOES, "nivel_minimo_alerta": NIVEL_MINIMO_ALERTA,
    "politica": POLITICA,
    "politica_codigo_sha256": sha256_arquivo(Path(politica_original.__file__)),
    "test_congelado_sha256": snapshot_test["sha256"],
    "escopo": "concordância com política visual; não validação independente de risco físico",
}
salvar_json(PASTA_SAIDA / "configuracao_execucao.json", configuracao)
salvar_json(PASTA_SAIDA / "manifesto_test.json", manifesto)

# Um YAML de avaliação separado resolve caminhos sem editar o data.yaml original.
yaml_avaliacao = {"path": BASE_DATASET.as_posix(), "train": "images/train",
                  "val": "images/val", "test": "images/test", "names": NOMES}
CAMINHO_YAML_AVALIACAO = PASTA_SAIDA / "data_avaliacao.yaml"
CAMINHO_YAML_AVALIACAO.write_text(yaml.safe_dump(yaml_avaliacao, allow_unicode=True,
                                              sort_keys=False), encoding="utf-8")
print("Pasta de resultados:", PASTA_SAIDA)


## 7. Avaliar o detector no TEST inteiro

O validador oficial calcula as métricas e os gráficos. No Windows, `workers=0` evita
problemas de multiprocessing no Jupyter. Em CPU, esta célula pode levar alguns minutos.
A pasta `metricas_detector_yolo/` guarda as curvas e a matriz de confusão do detector.

As métricas são números entre 0 e 1 nos CSVs. Multiplique por 100 para apresentar em
porcentagem. **mAP não é acurácia.** Comparações com o baseline devem observar a mesma
base, configuração de avaliação e versão do software.


In [ ]:
metricas = modelo.val(
    data=str(CAMINHO_YAML_AVALIACAO), split="test", imgsz=IMGSZ,
    batch=BATCH_AVALIACAO, device=DISPOSITIVO, workers=0,
    conf=0.001, iou=IOU_NMS, max_det=MAX_DETECCOES,
    plots=True, project=str(PASTA_SAIDA), name="metricas_detector_yolo",
    exist_ok=False, verbose=False,
)
metricas_detector = {
    "precision": float(metricas.box.mp), "recall": float(metricas.box.mr),
    "map50": float(metricas.box.map50), "map50_95": float(metricas.box.map),
}
pd.DataFrame([metricas_detector]).to_csv(PASTA_SAIDA / "metricas_detector.csv",
                                       index=False, encoding="utf-8-sig")
por_classe = []
for indice, classe_id in enumerate(metricas.box.ap_class_index):
    precision, recall, ap50, ap = metricas.box.class_result(indice)
    por_classe.append({"classe": NOMES[int(classe_id)], "precision": float(precision),
                       "recall": float(recall), "map50": float(ap50), "map50_95": float(ap)})
pd.DataFrame(por_classe).to_csv(PASTA_SAIDA / "metricas_detector_por_classe.csv",
                              index=False, encoding="utf-8-sig")
display(pd.DataFrame([metricas_detector]).mul(100).round(2))


## 8. Gerar mensagens e avaliar o efeito das detecções sobre os alertas

Há duas passagens de inferência: a anterior calcula mAP em confiança baixa; esta usa
a confiança fixa da aplicação. Nenhum caso é removido porque o modelo errou.

Os tempos incluem a chamada de previsão e leitura da imagem, após uma chamada de
aquecimento. Não são medidas do tempo de treinamento nem garantia de desempenho em câmera.


In [ ]:
prever_imagem(modelo, REGISTROS[0]["imagem"])  # aquecimento, fora das medidas
linhas_objetos, linhas_imagens = [], []
predicoes = {}

for n, registro in enumerate(REGISTROS, 1):
    inicio = time.perf_counter()
    resultado = prever_imagem(modelo, registro["imagem"])
    tempo_ms = (time.perf_counter() - inicio) * 1000
    largura, altura = registro["largura"], registro["altura"]
    gt = adicionar_prioridades(registro["gt"], largura, altura)
    pred = adicionar_prioridades(objetos_resultado(resultado), largura, altura)
    predicoes[registro["nome"]] = pred
    pares, nao_detectados, nao_associados = corresponder_objetos(gt, pred)

    for gi, pi, iou in pares:
        linhas_objetos.append(linha_objeto(registro["nome"], "correspondencia",
                                          gi, pi, gt[gi], pred[pi], iou))
    for gi in nao_detectados:
        linhas_objetos.append(linha_objeto(registro["nome"], "objeto_nao_detectado",
                                          gi=gi, g=gt[gi]))
    for pi in nao_associados:
        linhas_objetos.append(linha_objeto(registro["nome"], "previsao_nao_associada",
                                          pi=pi, p=pred[pi]))

    mensagens = [gerar_mensagem(o) for o in pred if aciona_alerta(o["prioridade"])]
    prioridade_ref, prioridade_pred = prioridade_imagem(gt), prioridade_imagem(pred)
    linhas_imagens.append({
        "imagem": registro["nome"], "objetos_gt": len(gt), "objetos_pred": len(pred),
        "correspondencias": len(pares), "nao_detectados": len(nao_detectados),
        "previsoes_nao_associadas": len(nao_associados),
        "prioridade_ref_politica": prioridade_ref, "prioridade_pred": prioridade_pred,
        "ha_alerta_ref_politica": any(aciona_alerta(o["prioridade"]) for o in gt),
        "ha_alerta_pred": bool(mensagens), "mensagem_alerta": "\n".join(mensagens),
        "tempo_previsao_ms": round(tempo_ms, 3),
    })
    if n % 20 == 0 or n == len(REGISTROS):
        print(f"Processadas {n}/{len(REGISTROS)} imagens")

COLUNAS_OBJETOS = ["imagem", "evento", "id_gt", "id_pred", "classe_ref", "classe_pred",
                   "iou", "confianca", "bbox_ref", "bbox_pred", "prioridade_ref_politica",
                   "prioridade_pred", "ha_alerta_ref_politica", "ha_alerta_pred",
                   "concordancia_prioridade"]
df_objetos = pd.DataFrame(linhas_objetos, columns=COLUNAS_OBJETOS)
df_imagens = pd.DataFrame(linhas_imagens)
df_objetos.to_csv(PASTA_SAIDA / "avaliacao_objetos.csv", index=False, encoding="utf-8-sig")
df_imagens.to_csv(PASTA_SAIDA / "alertas_por_imagem.csv", index=False, encoding="utf-8-sig")
blocos_texto = [f"{r['imagem']}\n{r['mensagem_alerta']}" for r in linhas_imagens if r["mensagem_alerta"]]
(PASTA_SAIDA / "alertas_texto.txt").write_text("\n\n".join(blocos_texto), encoding="utf-8")
print("Mensagens e diagnósticos gerados automaticamente.")


## 9. Métricas dos alertas contra a política automática

Por objeto, um alerta correto nesta avaliação exige correspondência de classe/caixa
e acionamento tanto na referência da política quanto na previsão. Um objeto anotado
que exigia alerta e não foi detectado entra como omissão. Um alerta extra sem objeto
correspondente entra como falso alerta **em relação à política**, não como risco humano.

A análise por imagem avalia se o sistema emitiu algum alerta; ela complementa a análise
por objeto, pois uma imagem com vários obstáculos pode ter um alerta presente e ainda
assim conter obstáculos não detectados. `None` significa métrica sem denominador.


In [ ]:
def dividir_ou_nulo(numerador, denominador):
    return float(numerador / denominador) if denominador else None

def contagens_alertas(tabela):
    ref = tabela["ha_alerta_ref_politica"].astype(bool)
    prev = tabela["ha_alerta_pred"].astype(bool)
    tp = int((ref & prev).sum())
    fn = int((ref & ~prev).sum())
    fp = int((~ref & prev).sum())
    return {"tp_vs_politica": tp, "fn_vs_politica": fn, "fp_vs_politica": fp,
            "precision_vs_politica": dividir_ou_nulo(tp, tp + fp),
            "recall_vs_politica": dividir_ou_nulo(tp, tp + fn),
            "f1_vs_politica": dividir_ou_nulo(2 * tp, 2 * tp + fp + fn)}

correspondencias = df_objetos["evento"].eq("correspondencia")
ref_relevante = df_objetos["prioridade_ref_politica"].ne("sem_prioridade")
mascara_concordancia = correspondencias & ref_relevante
ref_alta = df_objetos["prioridade_ref_politica"].eq("alto")

diagnostico = {
    "escopo": "referência por política fixa nas anotações; não mede risco real independentemente",
    "imagens": len(df_imagens),
    "objetos_anotados": int(df_imagens["objetos_gt"].sum()),
    "objetos_previstos": int(df_imagens["objetos_pred"].sum()),
    "diagnostico_detector_conf_fixa": {
        "tp_iou50_mesma_classe": int(correspondencias.sum()),
        "fn_iou50_mesma_classe": int(df_objetos["evento"].eq("objeto_nao_detectado").sum()),
        "fp_iou50_mesma_classe": int(df_objetos["evento"].eq("previsao_nao_associada").sum()),
    },
    "alertas_por_objeto_vs_politica": contagens_alertas(df_objetos),
    "alertas_por_imagem_vs_politica": contagens_alertas(df_imagens),
    "concordancia_prioridade_em_objetos_correspondentes": dividir_ou_nulo(
        int(df_objetos.loc[mascara_concordancia, "concordancia_prioridade"].eq(True).sum()),
        int(mascara_concordancia.sum())),
    "concordancia_prioridade_por_imagem": float(
        df_imagens["prioridade_ref_politica"].eq(df_imagens["prioridade_pred"]).mean()),
    "objetos_prioridade_alta_ref_sem_alerta": int(
        (ref_alta & ~df_objetos["ha_alerta_pred"].astype(bool)).sum()),
    "tempo_previsao_medio_ms": float(df_imagens["tempo_previsao_ms"].mean()),
}
salvar_json(PASTA_SAIDA / "metricas_alertas_vs_politica.json", diagnostico)

por_classe_alertas = []
for classe in POLITICA["pesos_classe"]:
    tabela = df_objetos[df_objetos["classe_ref"].eq(classe)
                       | df_objetos["classe_pred"].eq(classe)]
    por_classe_alertas.append({"classe": classe, **contagens_alertas(tabela)})
pd.DataFrame(por_classe_alertas).to_csv(PASTA_SAIDA / "alertas_por_classe_vs_politica.csv",
                                     index=False, encoding="utf-8-sig")
print(json.dumps(diagnostico, ensure_ascii=False, indent=2))


## 10. Gráfico, resumo e conferência de preservação do TEST

O gráfico cruza a prioridade máxima da imagem na referência automática e na previsão.
Os objetos não detectados permanecem no diagnóstico, mesmo quando outro objeto da
mesma imagem já produziu um alerta. Nenhuma imagem precisa ser classificada por você.

Ao final, os hashes das imagens e labels são conferidos novamente. O validador YOLO
pode criar arquivos auxiliares `.cache`; esses arquivos não são anotações editadas.


In [ ]:
ordem = ["sem_prioridade", "baixo", "medio", "alto"]
matriz = pd.crosstab(df_imagens["prioridade_ref_politica"], df_imagens["prioridade_pred"])
matriz = matriz.reindex(index=ordem, columns=ordem, fill_value=0)
matriz.to_csv(PASTA_SAIDA / "matriz_prioridade_por_imagem.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 5))
cores = ax.imshow(matriz.to_numpy(), cmap="Blues")
rotulos = ["Sem prioridade", "Baixa", "Média", "Alta"]
ax.set_xticks(range(4), rotulos, rotation=20)
ax.set_yticks(range(4), rotulos)
ax.set_xlabel("Prioridade prevista")
ax.set_ylabel("Prioridade da política nas anotações")
ax.set_title("Concordância com a política visual — por imagem")
for linha in range(4):
    for coluna in range(4):
        valor = int(matriz.iloc[linha, coluna])
        cor = "white" if valor > matriz.to_numpy().max() / 2 else "black"
        ax.text(coluna, linha, valor, ha="center", va="center", color=cor)
fig.colorbar(cores, ax=ax, label="Imagens")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "matriz_prioridade_por_imagem.png", dpi=160)
plt.show()
plt.close(fig)

for item in manifesto:
    relativo = Path(item["imagem"])
    if (sha256_arquivo(PASTA_IMAGENS / relativo) != item["imagem_sha256"]
            or sha256_arquivo(PASTA_LABELS / relativo.with_suffix(".txt")) != item["label_sha256"]):
        raise RuntimeError(f"Arquivo de TEST foi modificado durante a execução: {relativo}")

resumo = (
    "AVALIAÇÃO AUTOMÁTICA DE DETECÇÃO E ALERTAS EM TEXTO\n"
    f"Imagens: {len(df_imagens)}\n"
    f"Objetos anotados: {diagnostico['objetos_anotados']}\n"
    f"Precision detector: {metricas_detector['precision']:.4f}\n"
    f"Recall detector: {metricas_detector['recall']:.4f}\n"
    f"mAP@50: {metricas_detector['map50']:.4f}\n"
    f"mAP@50-95: {metricas_detector['map50_95']:.4f}\n"
    "Alertas avaliados contra política fixa aplicada às anotações.\n"
    "A avaliação não comprova risco físico real nem generalização da regra.\n"
    "Imagens e labels do TEST conferidos por SHA-256, sem alterações.\n"
    "Não há leitura de planilha manual nem geração de áudio.\n"
)
(PASTA_SAIDA / "resumo.txt").write_text(resumo, encoding="utf-8")
print(resumo)
print("Resultados:", PASTA_SAIDA)
display(df_imagens[["imagem", "prioridade_pred", "mensagem_alerta"]].head(10))


## 11. Demonstrar a saída final em uma imagem nova — opcional

Esta célula é somente uma demonstração da aplicação. A imagem nova não entra nas métricas
porque não possui necessariamente anotações de referência. Informe o caminho se quiser
testar; deixe `None` para não executar. A ausência de mensagem significa apenas que a
política não acionou um alerta para as detecções obtidas.


In [ ]:
IMAGEM_NOVA = None  # por exemplo Path(r"C:\Users\Diego\Pictures\sala.jpg")

if IMAGEM_NOVA is not None:
    caminho = Path(IMAGEM_NOVA)
    if not caminho.is_file():
        raise FileNotFoundError(caminho)
    with Image.open(caminho) as imagem:
        largura, altura = imagem.size
    resultado_novo = prever_imagem(modelo, caminho)
    objetos_novos = adicionar_prioridades(objetos_resultado(resultado_novo), largura, altura)
    mensagens = [gerar_mensagem(o) for o in objetos_novos if aciona_alerta(o["prioridade"])]
    print("\n".join(mensagens) if mensagens else "Nenhum alerta acionado pelas regras configuradas.")


## Depois da execução

- O fluxo atual passa a ser: preparação da base → treinamento → **este notebook** → aplicação
  com mensagem em texto. Os notebooks manuais antigos ficam guardados como histórico.
- `avaliacao_objetos.csv` e `alertas_por_imagem.csv` são relatórios prontos; não complete colunas.
- O detector expandido poderá usar este mesmo notebook. Mude somente `CAMINHO_MODELO`,
  preservando as classes, o TEST e a configuração necessária à comparação.
- Se o conjunto de teste ou o mapeamento de classes não corresponder ao contexto, resolva
  a divergência na estrutura dos arquivos antes de continuar. Não crie gabarito por palpite.
- Para o TCC, relate separadamente as **métricas de detecção** e a **concordância dos alertas
  com a política visual**. Os resultados manuais dos 50 cenários pertencem ao experimento
  anterior e têm metodologia diferente.

O notebook foi entregue sem resultados executados no seu modelo. As verificações de código
e de lógica usam casos sintéticos; os resultados reais serão produzidos na sua execução.
